# Replicant — Convergence Benchmark Analysis

Loads `results/results.csv` and produces thesis-ready figures in `figures/`.

**Generate results first** (release build — debug timings carry tens of ms of overhead that swamps real signal):
```sh
mkdir -p results
cargo run --release --bin orchestrator -- --trials 10 --output csv \
  scenarios/*.toml \
  2>/dev/null > results/results.csv
```

> **If regenerating after a code change:** the CSV is cached as `results/results.parquet` for
> faster reruns. The cache is refreshed automatically when `results/results.csv` is newer than
> `results/results.parquet`. To force a rebuild, delete `results/results.parquet` manually.

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 150

REPO    = Path("..").resolve()  # absolute repo root; works regardless of launch CWD
RESULTS = REPO / "results"
CSV     = RESULTS / "results.csv"
PARQUET = RESULTS / "results.parquet"
FIGS    = REPO / "analysis" / "figures"
FIGS.mkdir(exist_ok=True)

## Load data

On first run the CSV is parsed and cached as Parquet (preserves dtypes, loads faster on reruns).

In [ ]:
csv_mtime = CSV.stat().st_mtime if CSV.exists() else 0
parquet_fresh = PARQUET.exists() and PARQUET.stat().st_mtime >= csv_mtime

if parquet_fresh:
    df = pd.read_parquet(PARQUET)
else:
    df = pd.read_csv(CSV)
    df.to_parquet(PARQUET)

trials  = df[df.row_type == "trial"].copy()
summary = df[df.row_type == "summary"].copy()
# In summary rows the `trial` column holds the trial count, not a trial number.
summary = summary.rename(columns={"trial": "n_trials"})

print(f"{len(trials)} trial records, {trials.scenario.nunique()} scenarios, "
      f"{trials.groupby('scenario').size().iloc[0]} trials each")
summary[["scenario", "n_trials", "node_count", "op_count", "mean_ms", "p50_ms", "p95_ms"]]

## Full-mesh: convergence vs node count

Measures time from last write to all-nodes fingerprint agreement. Split by write pattern: `concentrated` (all ops at node-0) vs `round_robin` (writes distributed across all nodes). Shaded band shows p50–p95 range across trials.

Concentrated wins across all N: with one author, replicas integrate every change in the same order, so peer-by-peer sync states converge in a single wave instead of reconciling the divergence that scattered writes create.

In [ ]:
mesh = summary[summary.scenario.str.startswith("full-mesh")].copy()
mesh["write_pattern"] = mesh.scenario.apply(
    lambda s: "concentrated" if s.endswith("-concentrated") else "round_robin"
)
mesh = mesh.sort_values(["write_pattern", "node_count"])

fig, ax = plt.subplots(figsize=(6, 4))
for pat, group in mesh.groupby("write_pattern"):
    ax.plot(group.node_count, group.mean_ms, marker="o", label=f"{pat} (mean)")
    ax.fill_between(group.node_count, group.p50_ms, group.p95_ms, alpha=0.15)
ax.set_xlabel("Node count")
ax.set_ylabel("Convergence (ms)")
ax.set_title("Full-mesh: convergence latency vs N — by write pattern")
ax.legend()
fig.tight_layout()
fig.savefig(FIGS / "full_mesh_scaling.pdf")
plt.show()

## Partition-heal: convergence vs partition depth

Measures time from heal trigger (cross-group edges added) to global convergence.
`ConnectPeer` blocks until the Automerge sync handshake is open (via an internal
readiness signal), so these numbers reflect actual CRDT merge cost rather than
any fixed settle delay.

Split by **two** dimensions:
- **`write_pattern`** (`round_robin` vs `concentrated`) — how writes were distributed during the partition phase.
- **`heal_topology`** (`full_mesh` vs `bridge`) — what wiring is added on heal. `full_mesh` reconnects every pair across the two groups; `bridge` adds only `groups[0].nodes[0] ↔ groups[1].nodes[0]`, forcing all cross-partition state through one edge.

The bridge variant is where the thesis story is loudest: with one cross-edge it has a longer diameter (3 hops) but vastly fewer edges to flood (3-13 vs 6-28 in full-mesh-heal). Convergence is **~7-21× faster** than full-mesh heal — relay amplification dominates over diameter even on the heal path. This is the same mechanism that makes line-n10 beat full-mesh-n10 in the steady-state experiments, applied to partition recovery.

In [ ]:
heal = summary[summary.scenario.str.startswith("partition")].copy()
heal["write_pattern"] = heal.scenario.apply(
    lambda s: "concentrated" if "concentrated" in s else "round_robin"
)
heal["heal_topology"] = heal.scenario.apply(
    lambda s: "bridge" if "bridge" in s else "full_mesh"
)
heal["depth"] = heal.node_count.astype(int)
heal = heal.sort_values(["depth", "heal_topology", "write_pattern"])

depths = sorted(heal.depth.unique())
# Series order keeps full_mesh on the left of each cluster so the bridge
# speedup reads visually as a step down between the two groups.
series = [
    ("full_mesh", "round_robin"),
    ("full_mesh", "concentrated"),
    ("bridge", "round_robin"),
    ("bridge", "concentrated"),
]
n_series = len(series)
width = 0.78 / n_series

fig, ax = plt.subplots(figsize=(9, 4.5))
for i, (heal_top, pat) in enumerate(series):
    sub = (
        heal[(heal.heal_topology == heal_top) & (heal.write_pattern == pat)]
        .set_index("depth")
        .reindex(depths)
    )
    offsets = [d + (i - (n_series - 1) / 2) * width for d in depths]
    yerr = (sub.p95_ms - sub.mean_ms).clip(lower=0).fillna(0)
    ax.bar(
        offsets,
        sub.mean_ms.fillna(0),
        width=width,
        yerr=yerr,
        capsize=3,
        label=f"{heal_top} / {pat}",
        alpha=0.85,
    )
ax.set_xticks(depths)
ax.set_xticklabels([f"n={d}" for d in depths])
ax.set_xlabel("Total nodes (2 partitions)")
ax.set_ylabel("Heal convergence (ms)")
ax.set_title("Partition-heal: convergence by heal_topology × write_pattern")
ax.legend(title="heal_topology / write_pattern", loc="upper left", fontsize=9)
fig.tight_layout()
fig.savefig(FIGS / "partition_heal.pdf")
plt.show()

## Convergence vs N — by topology kind

Each topology family scales differently. Restricted to `round_robin` writes so the topology comparison isn't confounded by write pattern; `concentrated` variants appear in the diameter plot below. Error bars span p50–p95.

In [ ]:
topo = summary[
    (~summary.scenario.str.endswith("-concentrated"))
    & (summary.topology_kind.isin(["full_mesh", "ring", "line", "star"]))
].sort_values(["topology_kind", "node_count"])

fig, ax = plt.subplots(figsize=(7, 4.5))
for kind, group in topo.groupby("topology_kind"):
    yerr_low = (group.mean_ms - group.p50_ms).clip(lower=0)
    yerr_high = (group.p95_ms - group.mean_ms).clip(lower=0)
    ax.errorbar(
        group.node_count,
        group.mean_ms,
        yerr=[yerr_low, yerr_high],
        marker="o",
        capsize=3,
        label=kind,
    )
ax.set_xlabel("Node count")
ax.set_ylabel("Convergence (ms)")
ax.set_title("Convergence vs N — by topology kind (round_robin)")
ax.legend(title="Topology")
fig.tight_layout()
fig.savefig(FIGS / "convergence_vs_n_by_topology.pdf")
plt.show()

## Convergence vs diameter — the structural correlate

Diameter is the longest shortest-path in the topology — the minimum number of hops any state must traverse to fully propagate. Conventional wisdom predicts `convergence_ms ~ diameter`, but Phase A's relay-on-receive amplifies with **edge count** and **max degree** too.

Below scatters every non-partition scenario by `(diameter, mean_ms)`, coloured by topology kind and shaped by write pattern. **Diameter alone does not predict convergence**: full-mesh (diameter = 1) ranges from <1 ms (n = 2) to >40 ms (n = 10) because edge count grows with N²; line-n10 (diameter 9) converges faster than full-mesh-n10 (diameter 1) because the line has only 9 edges vs 45.

This is the thesis's negative result: diameter is necessary (no traversal can be shorter) but not sufficient for predicting CRDT convergence under a relay-flooded sync layer.

In [ ]:
import matplotlib.patches as mpatches

diam_df = summary[summary.topology_kind != "partition_heal"].copy()
diam_df["write_pattern"] = diam_df.scenario.apply(
    lambda s: "concentrated" if s.endswith("-concentrated") else "round_robin"
)

kinds = sorted(diam_df.topology_kind.unique())
palette = dict(zip(kinds, sns.color_palette("muted", n_colors=len(kinds))))
markers = {"round_robin": "o", "concentrated": "s"}

fig, ax = plt.subplots(figsize=(8, 5))
for (kind, pat), group in diam_df.groupby(["topology_kind", "write_pattern"]):
    ax.scatter(
        group.diameter,
        group.mean_ms,
        marker=markers[pat],
        s=90,
        color=palette[kind],
        edgecolor="black",
        linewidth=0.5,
        alpha=0.85,
    )

for _, row in diam_df.iterrows():
    ax.annotate(
        f"n={row.node_count}",
        (row.diameter, row.mean_ms),
        textcoords="offset points",
        xytext=(6, 4),
        fontsize=8,
        color="dimgray",
    )

color_legend = [mpatches.Patch(color=palette[k], label=k) for k in kinds]
shape_legend = [
    plt.Line2D([0], [0], marker=markers[pat], color="black", linestyle="",
               markerfacecolor="lightgray", markersize=10, label=pat)
    for pat in markers
]
first_legend = ax.legend(handles=color_legend, title="Topology",
                          loc="upper left", bbox_to_anchor=(1.02, 1))
ax.add_artist(first_legend)
ax.legend(handles=shape_legend, title="Write pattern",
          loc="upper left", bbox_to_anchor=(1.02, 0.55))

ax.set_xlabel("Diameter (hops)")
ax.set_ylabel("Convergence (ms)")
ax.set_title("Convergence vs diameter — kind + write pattern matter too")
fig.tight_layout()
fig.savefig(FIGS / "convergence_vs_diameter.pdf", bbox_inches="tight")
plt.show()

## Raw distributions (box plots)

Shows the full trial distribution rather than summary statistics.
More honest for small N — outliers are visible.

In [ ]:
order = (
    trials.groupby("scenario")["convergence_ms"]
    .median()
    .sort_values()
    .index
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=trials, x="scenario", y="convergence_ms", order=order, ax=ax)
ax.tick_params(axis="x", rotation=25)
ax.set_xlabel(None)
ax.set_ylabel("Convergence (ms)")
ax.set_title("Convergence distribution per scenario")
fig.tight_layout()
fig.savefig(FIGS / "boxplot.pdf")
plt.show()

## Summary table

Formatted for copy-paste into the thesis evaluation section.

In [ ]:
table = summary[["scenario", "node_count", "n_trials", "mean_ms", "p50_ms", "p95_ms"]].copy()
table.columns = ["Scenario", "Nodes", "Trials", "Mean (ms)", "p50 (ms)", "p95 (ms)"]
table = table.set_index("Scenario")
table.round(1)

## Measurement stability

Standard deviation and coefficient of variation (CV = σ/μ) across trials.
CV < 10% is generally stable; CV > 30% suggests the measurement is noisy
and more trials or a quieter environment are needed.

In [ ]:
stability = (
    trials.groupby("scenario")["convergence_ms"]
    .agg(mean="mean", std="std", n="count")
    .assign(cv_pct=lambda df: (df["std"] / df["mean"] * 100).round(1))
    .round({"mean": 2, "std": 2})
    .sort_values("cv_pct", ascending=False)
)
stability.columns = ["Mean (ms)", "Std (ms)", "N trials", "CV (%)"]

# Highlight rows where CV exceeds 20% — worth investigating
def highlight_cv(row):
    return ["background-color: #fdd" if row["CV (%)"] > 20 else "" for _ in row]

stability.style.apply(highlight_cv, axis=1)

## OTel Protocol Metrics

Loaded from per-scenario `results/metrics-<scenario>.json` files written by the orchestrator when `--metrics-file` is passed.

OTel counters accumulate across every scenario in a single run, so **run one scenario at a time** to get per-scenario metrics. The loop below covers every TOML in `scenarios/`:

```sh
mkdir -p results
for s in $(ls scenarios/*.toml | xargs -n1 basename -s .toml); do
  cargo run --release --bin orchestrator -- --trials 10 \
    --metrics-file "results/metrics-${s}.json" \
    "scenarios/${s}.toml" \
    > /dev/null 2>&1
done
```

The `Sync traffic per write op` cell below reads every `results/metrics-<scenario>.json` and produces a cross-topology comparison.

Point `METRICS` below at any single file (e.g. `RESULTS / "metrics-full-mesh-n5.json"`) to inspect one scenario's per-node breakdown.

> **Multi-scenario (cumulative) alternative** — useful only as a sanity check that overall protocol traffic looks reasonable; values cannot be attributed to individual scenarios:
> ```sh
> cargo run --release --bin orchestrator -- --trials 10 --output csv \
>   --metrics-file results/metrics-all.json \
>   scenarios/*.toml \
>   2>/dev/null > results/results.csv
> ```

In [ ]:
import json

# Point this at a single-scenario metrics file for per-scenario analysis,
# e.g. RESULTS / "metrics-full-mesh-n5.json".
# Use RESULTS / "metrics-all.json" for the cumulative multi-scenario view.
METRICS = RESULTS / "metrics.json"


def load_metrics(path: Path) -> dict[str, pd.DataFrame] | None:
    """Parse a metrics JSON Lines file into a dict of DataFrames keyed by metric name.

    Each line in the file is a JSON object ``{"metrics": [...]}``.
    Data points from multiple flushes are concatenated per metric name.
    Returns None if the file does not exist.
    """
    if not path.exists():
        print(f"[metrics] {path} not found — run the per-scenario loop above first.")
        return None

    rows: list[dict] = []
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    by_name: dict[str, list[dict]] = {}
    for record in rows:
        for m in record.get("metrics", []):
            by_name.setdefault(m["name"], []).extend(m["data_points"])

    return {name: pd.DataFrame(points) for name, points in by_name.items()}


metrics = load_metrics(METRICS)
if metrics:
    print("Loaded metrics:", list(metrics.keys()))
    for name, df_m in metrics.items():
        print(f"  {name}: {len(df_m)} data points, columns={list(df_m.columns)}")

### Sync message traffic

Total Automerge sync messages sent and received per node.
Shows how much protocol chatter each node generates — useful for arguing O(N²) scaling of the gossip layer.


In [ ]:
if metrics:
    tx = metrics["replicant.sync.messages.tx"].groupby("actor")["value"].sum().rename("tx")
    rx = metrics["replicant.sync.messages.rx"].groupby("actor")["value"].sum().rename("rx")
    traffic = pd.concat([tx, rx], axis=1).fillna(0).astype(int)
    # Sort nodes naturally (node-0, node-1, …)
    traffic = traffic.loc[sorted(traffic.index, key=lambda s: int(s.split("-")[1]))]

    fig, ax = plt.subplots(figsize=(max(4, len(traffic) * 0.9), 4))
    x = range(len(traffic))
    width = 0.35
    ax.bar([i - width / 2 for i in x], traffic["tx"], width, label="sent (tx)", alpha=0.8)
    ax.bar([i + width / 2 for i in x], traffic["rx"], width, label="received (rx)", alpha=0.8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(traffic.index)
    ax.set_xlabel("Node")
    ax.set_ylabel("Sync messages")
    ax.set_title("Sync message traffic per node (all scenarios, cumulative)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGS / "sync_traffic.pdf")
    plt.show()
    print(traffic.assign(total=traffic.tx + traffic.rx).sort_values("total", ascending=False))


## Sync traffic per write op — by scenario

Loads every per-scenario `metrics-<scenario>.json` file (one per scenario, generated by the loop in the markdown above) and sums `tx + rx` across all nodes. Dividing by `trials × op_count` yields **sync messages per write op**, comparable across scenarios with different N and op_count.

This quantifies the **amplification cost** of Phase A's relay-on-receive fix: more edges (higher topology connectivity) → more redundant fan-out per write. It explains why full-mesh-n10 (45 edges) converges slower than line-n10 (9 edges) despite the shorter diameter — the line has 10× fewer message paths to flood.

In [ ]:
import matplotlib.patches as mpatches

sync_rows = []
for scen in summary.scenario.unique():
    mpath = RESULTS / f"metrics-{scen}.json"
    m = load_metrics(mpath)
    if m is None or "replicant.sync.messages.tx" not in m:
        continue
    tx_total = int(m["replicant.sync.messages.tx"]["value"].sum())
    rx_total = int(m["replicant.sync.messages.rx"]["value"].sum())
    s_row = summary[summary.scenario == scen].iloc[0]
    sync_rows.append({
        "scenario": scen,
        "topology_kind": s_row.topology_kind,
        "node_count": int(s_row.node_count),
        "edge_count": int(s_row.edge_count),
        "diameter": int(s_row.diameter),
        "msgs_per_op": (tx_total + rx_total) / (s_row.n_trials * s_row.op_count),
    })
sync_df = pd.DataFrame(sync_rows).sort_values(["topology_kind", "node_count"]).reset_index(drop=True)

kinds = sorted(sync_df.topology_kind.unique())
palette = dict(zip(kinds, sns.color_palette("muted", n_colors=len(kinds))))
bar_colors = [palette[k] for k in sync_df.topology_kind]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(sync_df.scenario, sync_df.msgs_per_op,
       color=bar_colors, edgecolor="black", linewidth=0.4, alpha=0.9)
ax.tick_params(axis="x", rotation=45)
for label in ax.get_xticklabels():
    label.set_horizontalalignment("right")
ax.set_xlabel(None)
ax.set_ylabel("Sync messages per write op (tx + rx, all nodes)")
ax.set_title("Relay amplification cost by scenario")
legend_handles = [mpatches.Patch(color=palette[k], label=k) for k in kinds]
ax.legend(handles=legend_handles, title="Topology", loc="upper left")
fig.tight_layout()
fig.savefig(FIGS / "sync_vs_topology.pdf")
plt.show()
sync_df.round(2)

### Op application latency

Per-node mean latency for `map_put` operations (from the histogram `sum / count`).
Measures the cost of Automerge's local commit — independent of network.


In [ ]:
if metrics:
    op_df = metrics["replicant.op.duration"].copy()
    op_df["mean_ms"] = op_df["sum"] / op_df["count"]
    op_df = op_df.sort_values("actor", key=lambda s: s.map(lambda v: int(v.split("-")[1])))

    fig, ax = plt.subplots(figsize=(max(4, len(op_df) * 0.9), 4))
    bars = ax.bar(op_df["actor"], op_df["mean_ms"], alpha=0.8)
    if "min" in op_df.columns and "max" in op_df.columns:
        ax.errorbar(
            op_df["actor"],
            op_df["mean_ms"],
            yerr=[op_df["mean_ms"] - op_df["min"], op_df["max"] - op_df["mean_ms"]],
            fmt="none",
            color="black",
            capsize=4,
        )
    ax.set_xlabel("Node")
    ax.set_ylabel("Mean op latency (ms)")
    ax.set_title("Automerge map_put latency per node (min/mean/max)")
    fig.tight_layout()
    fig.savefig(FIGS / "op_latency.pdf")
    plt.show()
    print(op_df[["actor", "count", "min", "mean_ms", "max"]].to_string(index=False))


### Document size after convergence — per-scenario per-node

For every scenario, the table below lists the serialized Automerge document size on each replica at scenario-end. The orchestrator's convergence check already enforces the **CRDT invariant** — every replica must reach the same logical state, i.e. byte-equal `state_fingerprint()` (sorted change hashes). Without that, no metrics file would exist.

What this table shows is something a strict CRDT check *doesn't* catch: Automerge's `save()` byte output is **not canonical** across replicas with identical logical state. Two replicas with the same DAG and the same readable values can produce different byte counts because the encoding preserves change-list storage order, and that order depends on the order each replica integrated remote changes.

Empirically (locked in by `adapter::tests::save_bytes_not_canonical_across_converged_replicas`):

- **Concentrated writes (any topology, including partition-heal)**: spread = 0. With a single author per replica's local history — node-0 for the full-mesh/line/ring/star cases, `group.nodes[0]` of each group for partition-heal — replicas integrate every change in the same author order, and `save()` produces byte-identical output. This is structurally guaranteed, not stochastic.
- **Round-robin on multi-author topologies**: spread > 0, but small (a few percent). The size depends on stochastic receive timing: replicas at different graph positions see remote changes in different orders. The full-mesh case is *not* always 0 — at n=10 with 20 writes scattered across 10 authors there's enough interleaving for timing-induced reorders to surface in the encoding (~4-5% spread). Smaller-N full-mesh runs often round to spread=0 simply because the change count is small and the receive order happens to align.

The `spread_pct` column is the right way to read this: a few percent is the encoding-ordering effect; anything larger would indicate a real timing or sync bug worth investigating (asserted below at the 10% threshold).

Empty cells (`—`) mean that scenario uses fewer nodes than the maximum in the suite.

In [ ]:
from IPython.display import display

doc_rows = []
missing = []
for scen in summary.scenario.unique():
    mpath = RESULTS / f"metrics-{scen}.json"
    if not mpath.exists():
        missing.append(scen)
        continue
    m = load_metrics(mpath)
    if m is None or "replicant.doc.size_bytes" not in m:
        missing.append(scen)
        continue
    # The gauge is re-sampled on every local op and every sync_receive, so
    # each actor appears in many rows. Keep the last flush per actor — by
    # scenario-end the cluster is converged (fingerprints match), so that
    # sample is each replica's post-convergence save() length.
    latest = m["replicant.doc.size_bytes"].drop_duplicates("actor", keep="last")
    for _, r in latest.iterrows():
        doc_rows.append({"scenario": scen, "actor": r["actor"], "bytes": int(r["value"])})

if not doc_rows:
    print("[doc_size] no per-scenario metrics files found — re-run scenarios with --metrics-file.")
else:
    doc_long = pd.DataFrame(doc_rows)
    node_cols = sorted(doc_long.actor.unique(), key=lambda s: int(s.split("-")[1]))
    doc_wide = doc_long.pivot(index="scenario", columns="actor", values="bytes").reindex(columns=node_cols)
    doc_wide["min"] = doc_wide.min(axis=1).astype(int)
    doc_wide["max"] = doc_wide.max(axis=1).astype(int)
    doc_wide["spread"] = (doc_wide["max"] - doc_wide["min"]).astype(int)
    doc_wide["spread_pct"] = (doc_wide["spread"] / doc_wide["min"] * 100).round(1)

    # Soft check: a few percent is the Automerge save()-ordering effect
    # documented in the markdown above; anything much larger would indicate
    # a real sync/timing bug. The CRDT convergence invariant itself is the
    # fingerprint match enforced by the orchestrator — see the
    # `adapter::tests::save_bytes_not_canonical_across_converged_replicas`
    # test that locks this property in.
    SPREAD_PCT_THRESHOLD = 10.0
    suspicious = doc_wide[doc_wide["spread_pct"] > SPREAD_PCT_THRESHOLD]
    assert suspicious.empty, (
        f"doc_size spread exceeds {SPREAD_PCT_THRESHOLD}% in: "
        f"{suspicious[['min', 'max', 'spread', 'spread_pct']].to_dict('index')}"
    )

    if missing:
        print(f"[doc_size] skipped {len(missing)} scenarios with no metrics file: {missing}")
    n_canonical = int((doc_wide["spread"] == 0).sum())
    print(f"{n_canonical}/{len(doc_wide)} scenarios have byte-identical save() across replicas")
    print(f"max spread observed: {int(doc_wide['spread'].max())} bytes "
          f"({doc_wide['spread_pct'].max():.1f}%)\n")

    # display() instead of last-expression rendering — the table is nested
    # inside `else:` so Jupyter's auto-render doesn't fire. Sort by spread
    # descending so the non-canonical scenarios surface at the top.
    display(doc_wide.sort_values("spread", ascending=False).fillna("—"))

## Prometheus-backed metrics (live stack)

This section pulls metrics from a running Prometheus instance instead of the per-scenario JSON files above. The intended flow is:

```sh
docker compose -f deploy/docker/compose.yaml up -d --build
cargo run --release --bin orchestrator -- \
  --replicas localhost:50051=replica-0:50051,localhost:50052=replica-1:50051,\
localhost:50053=replica-2:50051,localhost:50054=replica-3:50051,localhost:50055=replica-4:50051 \
  scenarios/full-mesh-n5.toml
# ...inspect Prometheus at http://localhost:9090, then re-run this section...
docker compose -f deploy/docker/compose.yaml down
```

If Prometheus isn't reachable (no stack up), this section is skipped — the JSON-file sections above remain the primary path for offline thesis runs.

In [ ]:
import requests

PROM = "http://localhost:9090"


def prom_query(q: str, prom: str = PROM) -> list[dict]:
    """Run a PromQL instant query. Returns [] if Prometheus is unreachable."""
    try:
        r = requests.get(f"{prom}/api/v1/query", params={"query": q}, timeout=2)
        r.raise_for_status()
        data = r.json()
        return data["data"]["result"] if data.get("status") == "success" else []
    except (requests.RequestException, ValueError):
        return []


prom_alive = bool(prom_query("up"))
print(f"Prometheus at {PROM}: {'reachable' if prom_alive else 'unreachable — section skipped'}")

### Sync message traffic per actor

Total sent (`tx`) and received (`rx`) sync messages per actor, aggregated across all peers. Same view as the JSON-file section above, but live from the TSDB.

In [ ]:
if prom_alive:
    def _to_df(result, value_col):
        return pd.DataFrame(
            [{"actor": s["metric"]["actor"], value_col: int(float(s["value"][1]))} for s in result]
        )

    tx = _to_df(prom_query("sum by (actor) (replicant_sync_messages_tx_total)"), "tx")
    rx = _to_df(prom_query("sum by (actor) (replicant_sync_messages_rx_total)"), "rx")
    traffic = (
        tx.set_index("actor")
          .join(rx.set_index("actor"), how="outer")
          .fillna(0).astype(int)
    )
    traffic = traffic.loc[sorted(traffic.index, key=lambda s: int(s.split("-")[1]))]

    fig, ax = plt.subplots(figsize=(max(4, len(traffic) * 0.9), 4))
    x = range(len(traffic))
    width = 0.35
    ax.bar([i - width / 2 for i in x], traffic["tx"], width, label="sent (tx)", alpha=0.8)
    ax.bar([i + width / 2 for i in x], traffic["rx"], width, label="received (rx)", alpha=0.8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(traffic.index)
    ax.set_xlabel("Actor")
    ax.set_ylabel("Sync messages")
    ax.set_title("Sync message traffic per actor (from Prometheus)")
    ax.legend()
    fig.tight_layout()
    plt.show()
    print(traffic.assign(total=traffic.tx + traffic.rx))

### Document size — post-convergence (live stack)

Live equivalent of the per-scenario table above. After every `sync_receive` the replica re-samples `replicant.doc.size_bytes`, so each actor's latest gauge value reflects post-convergence state.

The CRDT convergence invariant (fingerprint equality) is enforced by the orchestrator; this cell surfaces the `save()` byte spread as a property of the encoding rather than asserting strict equality. For full-mesh scenarios — the only thing `just smoke-docker` runs today — spread is expected to be 0; if you point this at a non-mesh scenario you should expect a few-percent spread, per the analysis above.

In [ ]:
if prom_alive:
    doc = (
        pd.DataFrame(
            [
                {"actor": s["metric"]["actor"], "bytes": int(float(s["value"][1]))}
                for s in prom_query("replicant_doc_size_bytes")
            ]
        )
        .sort_values("actor", key=lambda s: s.map(lambda v: int(v.split("-")[1])))
        .reset_index(drop=True)
    )

    fig, ax = plt.subplots(figsize=(max(4, len(doc) * 0.9), 4))
    ax.bar(doc["actor"], doc["bytes"], alpha=0.8)
    ax.set_xlabel("Actor")
    ax.set_ylabel("Document size (bytes)")
    ax.set_title("Post-convergence document size per actor (from Prometheus)")
    fig.tight_layout()
    plt.show()

    spread = int(doc["bytes"].max() - doc["bytes"].min())
    spread_pct = (spread / int(doc["bytes"].min()) * 100) if len(doc) else 0.0
    print(doc.to_string(index=False))
    print(f"\nspread: {spread} bytes ({spread_pct:.1f}%)")
    # Soft check matching the per-scenario cell above. A few-percent spread
    # is the Automerge save()-ordering effect; anything much larger is a
    # real sync/timing bug.
    assert spread_pct <= 10.0, (
        f"doc_size spread {spread_pct:.1f}% exceeds 10% — investigate"
    )
    if spread == 0:
        print("✓ save() bytes identical across replicas (expected for full-mesh)")
    else:
        print(f"✓ save() bytes within expected encoding-ordering tolerance")